# PyCAM-SIMA complete-CAM interactive workflow

This is the single maintained PyCAM-SIMA Notebook. It controls the complete 24-rank CAM-SIMA model and exposes three Python interfaces:

- `model.options`: explicit settings fixed before `cam_init`;
- `model.step_plan`: the editable top-level CAM phase order used by `model.step()`;
- `model.parameters`: typed handles for reading and writing live CAM fields.

Choose the FKESSLER configuration for Kessler physics or FADIAB for a real SE dynamics-only run. Complete-CAM physics is selected during initialization; it is not disabled by silently skipping `cam_run2`.

## 1. Explicit configuration

Change `physics_profile` before running this cell. Runtime options may be edited until `model.start()` calls `cam_init`; changing them afterward requires a new session and fresh run directory.

In [1]:
from datetime import datetime
from pathlib import Path
import os
import shutil

import pycam_sima
from pycam_sima import (
    FullCAMRuntimeOptions,
    FullCAMStepPlan,
    NotebookSession,
)
from pycam_sima.config import CaseConfig

repo = Path("/glade/work/ruitong/pycam-sima")
scratch = Path(os.environ.get("SCRATCH", "/glade/derecho/scratch/ruitong"))
physics_profile = "kessler"  # Use "adiabatic" for real dynamics only.

profiles = {
    "kessler": {
        "config": repo / "configs/fkessler_ne3pg3.yaml",
        "case": repo / "reference/cases/FKESSLER_ne3pg3_gnu_24x50",
        "reference_run": scratch / "pycam-sima/FKESSLER_ne3pg3_gnu_24x50/FKESSLER_ne3pg3_gnu_24x50/run",
    },
    "adiabatic": {
        "config": repo / "configs/adiabatic_ne3pg3.yaml",
        "case": repo / "reference/cases/FADIAB_ne3pg3_gnu_24x50",
        "reference_run": scratch / "pycam-sima/FADIAB_ne3pg3_gnu_24x50/FADIAB_ne3pg3_gnu_24x50/run",
    },
}
selected = profiles[physics_profile]
config = CaseConfig.from_yaml(selected["config"])

options = FullCAMRuntimeOptions(
    timestep_seconds=1800,
    physics_profile=physics_profile,
    mediator_present=False,
)
step_plan = FullCAMStepPlan.default()

stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
run_dir = scratch / "pycam-sima/notebook_trials" / f"{physics_profile}-{stamp}" / "run"
run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(selected["reference_run"] / "atm_in", run_dir / "atm_in")

print("pycam_sima", pycam_sima.__version__)
print("run directory:", run_dir)
print("options:", options.describe())
step_plan.describe()

pycam_sima 0.5.0
run directory: /glade/derecho/scratch/ruitong/pycam-sima/notebook_trials/kessler-20260719-160955/run
options: {'timestep_seconds': 1800, 'physics_profile': 'kessler', 'mediator_present': False, 'physics_enabled': True, 'dynamics_enabled': True, 'mutable_until': 'cam_init'}


[{'order': 1,
  'name': 'cam_run2',
  'category': 'physics_and_mapping',
  'description': 'physics after coupler, then physics-to-dynamics mapping',
  'required': True,
  'enabled': True},
 {'order': 2,
  'name': 'cam_run3',
  'category': 'dynamics',
  'description': 'SE dynamics',
  'required': True,
  'enabled': True},
 {'order': 3,
  'name': 'cam_run4',
  'category': 'post_dynamics',
  'description': 'post-dynamics CAM work',
  'required': True,
  'enabled': True},
 {'order': 4,
  'name': 'cam_timestep_final',
  'category': 'lifecycle',
  'description': 'finish the current CAM timestep',
  'required': True,
  'enabled': True},
 {'order': 5,
  'name': 'advance_timestep',
  'category': 'clock',
  'description': 'advance the native CAM clock',
  'required': True,
  'enabled': True},
 {'order': 6,
  'name': 'cam_timestep_init',
  'category': 'lifecycle_and_mapping',
  'description': 'initialize the next timestep and map dynamics to physics',
  'required': True,
  'enabled': True},
 {'or

## 2. Start the complete MPI model

From a Derecho login-node kernel, `start()` submits a 24-rank PBS worker. Inside an allocation it launches locally. The cell returns only when all ranks finish initialization and wait for Python commands.

In [2]:
if "model" in globals() and model.running:
    model.close()

model = NotebookSession(
    config,
    run_dir=run_dir,
    env_script=selected["case"] / ".env_mach_specific.sh",
    python_executable=repo / ".venv/bin/python",
    log_path=run_dir / "mpi-worker.log",
    options=options,
    step_plan=step_plan,
)
model.start()
print(
    f"ready: mode={model.launch_mode_used}, job={model.job_id}, "
    f"ranks={model.ranks}, fields={len(model.field_names)}, step={model.current_step}"
)

PyCAM-SIMA PBS worker submitted as 6804831.desched1; waiting for 24 MPI ranks ...
ready: mode=pbs, job=6804831.desched1, ranks=24, fields=21, step=0


## 3. Inspect the Python control interfaces

The plan shown here is the actual plan sent to every MPI rank by `model.step()`. `parameters.describe()` lists the typed key fields and every additional field available through `parameters.field(name)`.

In [3]:
print("runtime options:", model.options.describe())
print("phase status:", model.phase_status)
print("step plan:")
display(model.step_plan.describe())
print("parameters and fields:")
model.parameters.describe()

runtime options: {'timestep_seconds': 1800, 'physics_profile': 'kessler', 'mediator_present': False, 'physics_enabled': True, 'dynamics_enabled': True, 'mutable_until': 'cam_init'}
phase status: {'last_phase': 'cam_run1', 'next_phase': 'cam_run2', 'sequence_safe': True, 'cycle_kind': 'initial_send', 'cycle_complete': True, 'step': 0, 'native_nstep': 0, 'plan_safe': True, 'plan_phases': ('cam_run2', 'cam_run3', 'cam_run4', 'cam_timestep_final', 'advance_timestep', 'cam_timestep_init', 'cam_run1'), 'plan_enabled': ('cam_run2', 'cam_run3', 'cam_run4', 'cam_timestep_final', 'advance_timestep', 'cam_timestep_init', 'cam_run1')}
step plan:


[{'order': 1,
  'name': 'cam_run2',
  'category': 'physics_and_mapping',
  'description': 'physics after coupler, then physics-to-dynamics mapping',
  'required': True,
  'enabled': True},
 {'order': 2,
  'name': 'cam_run3',
  'category': 'dynamics',
  'description': 'SE dynamics',
  'required': True,
  'enabled': True},
 {'order': 3,
  'name': 'cam_run4',
  'category': 'post_dynamics',
  'description': 'post-dynamics CAM work',
  'required': True,
  'enabled': True},
 {'order': 4,
  'name': 'cam_timestep_final',
  'category': 'lifecycle',
  'description': 'finish the current CAM timestep',
  'required': True,
  'enabled': True},
 {'order': 5,
  'name': 'advance_timestep',
  'category': 'clock',
  'description': 'advance the native CAM clock',
  'required': True,
  'enabled': True},
 {'order': 6,
  'name': 'cam_timestep_init',
  'category': 'lifecycle_and_mapping',
  'description': 'initialize the next timestep and map dynamics to physics',
  'required': True,
  'enabled': True},
 {'or

parameters and fields:


{'runtime': {'timestep_seconds': 1800,
  'physics_profile': 'kessler',
  'mediator_present': False,
  'physics_enabled': True,
  'dynamics_enabled': True,
  'mutable_until': 'cam_init'},
 'key_fields': {'air_temperature': {'shape': (27, 30),
   'dtype': '<f8',
   'dimensions': ('horizontal_dimension', 'vertical_layer_dimension'),
   'owner': 'native_view'},
  'eastward_wind': {'shape': (27, 30),
   'dtype': '<f8',
   'dimensions': ('horizontal_dimension', 'vertical_layer_dimension'),
   'owner': 'native_view'},
  'northward_wind': {'shape': (27, 30),
   'dtype': '<f8',
   'dimensions': ('horizontal_dimension', 'vertical_layer_dimension'),
   'owner': 'native_view'},
  'surface_air_pressure': {'shape': (27,),
   'dtype': '<f8',
   'dimensions': ('horizontal_dimension',),
   'owner': 'native_view'},
  'air_pressure_thickness': {'shape': (27, 30),
   'dtype': '<f8',
   'dimensions': ('horizontal_dimension', 'vertical_layer_dimension'),
   'owner': 'native_view'},
  'ccpp_constituents': {'

## 4. Read a live CAM field

The typed field handle makes the parameter name explicit. `get()` transfers a rank-local NumPy copy to the Notebook; `stats()` computes a compact summary on the MPI worker.

In [4]:
temperature_field = model.parameters.air_temperature
print(temperature_field.info)
print(temperature_field.stats(rank=0))
temperature = temperature_field.get(rank=0)
temperature

{'shape': (27, 30), 'dtype': '<f8', 'dimensions': ('horizontal_dimension', 'vertical_layer_dimension'), 'owner': 'native_view'}
{'rank': 0, 'shape': (27, 30), 'dtype': '<f8', 'min': 149.8407866754426, 'max': 306.29054325059553, 'mean': 237.51537148398492}


array([[150.20253396, 158.94337244, 167.16118088, 174.73367559,
        181.57244239, 187.71430852, 193.05934551, 197.28293981,
        201.03648614, 205.10012764, 209.52062187, 214.33856299,
        219.59632439, 225.33312797, 231.5994718 , 238.41764954,
        245.79321125, 253.68209419, 261.98531497, 269.87826244,
        276.48345972, 281.6156288 , 285.31699146, 287.64103729,
        289.22459305, 290.65616286, 291.93823274, 293.07193045,
        294.05730233, 294.89447155],
       [150.32655508, 159.19389267, 167.58283913, 175.34162418,
        182.34181255, 188.59546082, 193.99676679, 198.23268511,
        201.97220267, 205.99395396, 210.33777076, 215.03799062,
        220.13024923, 225.64641816, 231.62669153, 238.08725527,
        245.02996247, 252.4160406 , 260.16325502, 267.52475172,
        273.70461795, 278.53314474, 282.03501367, 284.24340996,
        285.75205094, 287.11841745, 288.34391967, 289.42887324,
        290.37274642, 291.17523502],
       [150.42552706, 159.3942

## 5. Advance one complete timestep

`step()` executes `model.step_plan` collectively on all 24 ranks, then all ranks return to the command wait loop. The default plan preserves the validated CAM order.

In [5]:
step = model.step()
print("completed step:", step)
print(temperature_field.stats(rank=0))

completed step: 1
{'rank': 0, 'shape': (27, 30), 'dtype': '<f8', 'min': 149.84036762606746, 'max': 306.30091247707594, 'mean': 237.5147291811104}


## 6. Pause after each top-level CAM phase

`run_phase()` sends exactly one call to every MPI rank. Re-run the next cell to walk through `cam_run2`, SE dynamics, finalization, clock advancement, initialization, and `cam_run1`.

In [6]:
phase = model.next_phase
status = model.run_phase(phase)
stats = temperature_field.stats(rank=0)
print(
    f"finished={phase} next={status['next_phase']} "
    f"step={status['step']} native_nstep={status['native_nstep']} "
    f"Tmean={stats['mean']:.17g}"
)

finished=cam_run2 next=cam_run3 step=1 native_nstep=2 Tmean=237.51472918109323


## 7. Optional targeted field modification

Field edits are allowed at every Python boundary. `set()` writes the supplied values into CAM memory on the selected rank. This intentionally breaks BFB.

In [ ]:
# changed = temperature_field.get(rank=0)
# changed[0, 0] += 0.01
# temperature_field.set(changed, rank=0)
# print(temperature_field.stats(rank=0))

## 8. Optional process on/off and ordering experiments

Every complete-CAM phase is required by the validated sequence. Disabling SE dynamics or changing order therefore requires `unsafe=True`. The modified plan is used by the next `model.step()` and may fail inside CAM if the requested sequence violates native lifecycle assumptions. After an unsafe step, restart CAM before returning to the default plan.

For a scientifically valid dynamics-only run, set `physics_profile = "adiabatic"` in section 1 and start a fresh FADIAB session.

In [7]:
# Turn off the real SE dynamics call for an explicit control experiment:
# model.step_plan.disable("cam_run3", unsafe=True)

# Or change the complete-CAM order:
# model.step_plan.move("cam_run3", before="cam_run2", unsafe=True)

# model.step()
model.step_plan.describe()

[{'order': 1,
  'name': 'cam_run2',
  'category': 'physics_and_mapping',
  'description': 'physics after coupler, then physics-to-dynamics mapping',
  'required': True,
  'enabled': True},
 {'order': 2,
  'name': 'cam_run3',
  'category': 'dynamics',
  'description': 'SE dynamics',
  'required': True,
  'enabled': True},
 {'order': 3,
  'name': 'cam_run4',
  'category': 'post_dynamics',
  'description': 'post-dynamics CAM work',
  'required': True,
  'enabled': True},
 {'order': 4,
  'name': 'cam_timestep_final',
  'category': 'lifecycle',
  'description': 'finish the current CAM timestep',
  'required': True,
  'enabled': True},
 {'order': 5,
  'name': 'advance_timestep',
  'category': 'clock',
  'description': 'advance the native CAM clock',
  'required': True,
  'enabled': True},
 {'order': 6,
  'name': 'cam_timestep_init',
  'category': 'lifecycle_and_mapping',
  'description': 'initialize the next timestep and map dynamics to physics',
  'required': True,
  'enabled': True},
 {'or

## 9. Optional complete 50-step run

Run this only with the unmodified default plan and no field edits if the goal is BFB validation.

In [8]:
# while model.current_step < config.steps:
#     model.step()
#     print(model.current_step, temperature_field.stats(rank=0)["mean"])

## 10. Finalize

Always close the session so CAM finalizes and the MPI worker exits.

In [9]:
model.close()
print("closed:", run_dir)

closed: /glade/derecho/scratch/ruitong/pycam-sima/notebook_trials/kessler-20260719-160955/run


## 11. BFB comparison

The comparison is intentionally fail-closed. A partial run reports `bfb=False` with missing files even when every available timestamp matches. A complete 50-step run should contain 51 files.

In [10]:
from pycam_sima.history_compare import compare_history

comparison = compare_history(selected["reference_run"], run_dir)
comparison.to_dict()

{'bfb': False,
 'reference_files': 51,
 'candidate_files': 2,
 'compared_files': 2,
 'missing_in_candidate': ('0001-01-01-03600.nc',
  '0001-01-01-05400.nc',
  '0001-01-01-07200.nc',
  '0001-01-01-09000.nc',
  '0001-01-01-10800.nc',
  '0001-01-01-12600.nc',
  '0001-01-01-14400.nc',
  '0001-01-01-16200.nc',
  '0001-01-01-18000.nc',
  '0001-01-01-19800.nc',
  '0001-01-01-21600.nc',
  '0001-01-01-23400.nc',
  '0001-01-01-25200.nc',
  '0001-01-01-27000.nc',
  '0001-01-01-28800.nc',
  '0001-01-01-30600.nc',
  '0001-01-01-32400.nc',
  '0001-01-01-34200.nc',
  '0001-01-01-36000.nc',
  '0001-01-01-37800.nc',
  '0001-01-01-39600.nc',
  '0001-01-01-41400.nc',
  '0001-01-01-43200.nc',
  '0001-01-01-45000.nc',
  '0001-01-01-46800.nc',
  '0001-01-01-48600.nc',
  '0001-01-01-50400.nc',
  '0001-01-01-52200.nc',
  '0001-01-01-54000.nc',
  '0001-01-01-55800.nc',
  '0001-01-01-57600.nc',
  '0001-01-01-59400.nc',
  '0001-01-01-61200.nc',
  '0001-01-01-63000.nc',
  '0001-01-01-64800.nc',
  '0001-01-01-666